## MSE Error

In [1]:
def mse(predicted, observed):
    n = len(predicted)
    squared_error = sum( (y_hat-y_i)**2 for y_hat, y_i in zip(predicted, observed))
    return squared_error/ n

In [2]:
y_true = [1, 2, 3]
y_pred = [1, 2, 13]
expected_mse = 100 / 3
print(expected_mse)
print(mse(y_pred, y_true))

33.333333333333336
33.333333333333336


In [13]:
# using pytorch
from torch import nn
import torch
y_hat = torch.tensor(y_true,  dtype=torch.float32)
y = torch.tensor(y_pred,  dtype=torch.float32)
torch_mse_loss = nn.MSELoss()
torch_mse_loss(y_hat, y)

tensor(33.3333)

## Cross Entropy Loss

In [14]:
def logit(out, sumOfLogits):
    return out/ sumOfLogits
    
def soft_max(outs):
    sumOfLogits = sum(outs)
    return [ logit(out, sumOfLogits) for out in outs ] 

In [15]:
import numpy as np

# Raw model outputs: logits
logits = np.array([
    [2.0, 1.0, 0.1],   # true class: 0
    [0.5, 2.5, 0.3],   # true class: 1
    [0.2, 0.1, 3.0],   # true class: 2
    [1.0, 2.0, 0.5],   # true class: 0
])

# Correct class indices
y_true = np.array([0, 1, 2, 0])

In [17]:
# were going to skip the straight python implementation and use np

def soft_max(z):
    shifted = z - np.max(z)
    logits = np.exp(shifted)
    return logits/ np.sum(logits)
    
def cross_entropy_loss(y_hat, y_pred):
    return -1* (np.dot(y_hat, np.log(y_pred)))
print( list(soft_max(z) for z in logits))

[array([0.65900114, 0.24243297, 0.09856589]), array([0.10860373, 0.80247906, 0.08891721]), array([0.05449744, 0.04931133, 0.89619123]), array([0.2312239 , 0.62853172, 0.14024438])]


#### Some notes:
The above works for classification as a one hot vector but in the case of doing batch examples we need to modify things in the following

In [ ]:
def softmax_batch(z):
    shifted = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(shifted)
    # normalize of axis ( in the case above np.max is for each column ) in other words
    #rows    = samples
    #columns = classes
    return exp_z / np.sum(exp_z, axis=1, keepdims=True) # keepdims is to make sure shape is preserved


def cross_entropy_loss_batch(y_true, y_pred):
    eps = 1e-15 # to avoid log(0)
    y_pred = np.clip(y_pred, eps, 1 - eps) # basically explicitly set bounds/ range for the pred tensor
    losses = -np.sum(y_true * np.log(y_pred), axis=1)
    return np.mean(losses) # we use mean here because we need to change the vector of losses to scalar - "on average how off am i per example"